In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
ticker = "EQNR.OL"
start = "2000-01-01"
end = None

In [3]:
df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)

if df.empty:
    raise ValueError(f"Ingen data for {ticker} i perioden.")
    
px = df["Adj Close"].dropna() #.rename(columns={"Adj Close":"price"})
px.columns = ["price"]
px = px.reset_index()
px

,Date,price
0,2000-01-03,22.395346
1,2000-01-04,22.037804
2,2000-01-05,21.582748
3,2000-01-06,22.362843
4,2000-01-07,22.720387
...,...,...
6627,2026-02-19,279.200012
6628,2026-02-20,273.700012
6629,2026-02-23,277.500000
6630,2026-02-24,280.200012


In [6]:
# Autoregression diagnostics and test
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.stats.diagnostic import acorr_ljungbox

# Build a clean series from the downloaded DataFrame (use Adj Close)
if 'df' in globals():
    series = df['Adj Close'].dropna().sort_index()
else:
    raise RuntimeError('DataFrame `df` not found - run the download cell first')

print('Series length:', len(series))

# 1) Augmented Dickey-Fuller test for stationarity
adf_res = adfuller(series)
print('ADF Statistic:', adf_res[0])
print('p-value:', adf_res[1])
print('Used lags:', adf_res[2])
print('Number of observations used for ADF regression:', adf_res[3])
for k, v in adf_res[4].items():
    print('Critial value (', k, '):', v)

# 2) Plot ACF and PACF to visually inspect serial correlation
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
plot_acf(series, lags=40, ax=axes[0])
plot_pacf(series, lags=40, ax=axes[1], method='ywm')
axes[0].set_title('ACF')
axes[1].set_title('PACF')
plt.tight_layout()
plt.show()

# 3) Fit AutoReg models for a range of lags and choose by AIC
best_aic = float('inf')
best_lag = None
best_model = None
max_lag = min(24, int(len(series)//5))
for p in range(1, max(2, max_lag+1)):
    try:
        model = AutoReg(series, lags=p, old_names=False).fit()
        if model.aic < best_aic:
            best_aic = model.aic
            best_lag = p
            best_model = model
    except Exception:
        pass

print(f'Selected AR lag: {best_lag} (AIC={best_aic:.2f})')
if best_model is not None:
    display(best_model.summary())

# 4) Ljung-Box test on residuals to check remaining autocorrelation
if best_model is not None:
    lb = acorr_ljungbox(best_model.resid, lags=[10, 20], return_df=True)
    print('Ljung-Box test of residuals:')
    print(lb)

# 5) Quick suggestion: use returns for a stationary series if price is non-stationary
if adf_res[1] > 0.05:
    print('ADF indicates non-stationarity (p>0.05). Consider using returns or differences:')
    print('series.diff().dropna() or np.log(series).diff().dropna()')

ModuleNotFoundError: No module named 'statsmodels'